In [ ]:
# --- Instalações ---
# (O 'imblearn' é para o RandomUnderSampler)
!pip install -q scikit-learn pandas numpy tensorflow imbalanced-learn

# --- Imports de Sistema ---
import pandas as pd
import numpy as np
import sys
import os
import warnings

# --- Imports de Deep Learning (Keras/TensorFlow) ---
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Embedding, Dropout, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping # <-- Novo! Para combater overfitting

# --- Imports de Machine Learning (Sklearn) ---
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.exceptions import UndefinedMetricWarning

# --- Imports de Balanceamento ---
from imblearn.under_sampling import RandomUnderSampler

# Ignora warnings
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# --- Montar o Google Drive ---
print("Montando Google Drive...")
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print(f"Drive já montado ou erro: {e}")

# --- Confirmação de GPU ---
print("\nVerificando GPU...")
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  print(
      '\n\nATENÇÃO: GPU NÃO ENCONTRADA! '
      'Vá em "Ambiente de execução" -> "Alterar o tipo de ambiente de execução" e selecione "GPU (T4)".'
  )
else:
  print(f'GPU encontrada: {device_name}')

Montando Google Drive...
Mounted at /content/drive

Verificando GPU...
GPU encontrada: /device:GPU:0


In [ ]:
# --- 1. Configurações ---

# --- ATENÇÃO: Verifique este caminho! ---
GDRIVE_PATH = '/content/drive/MyDrive/Faculdade/IC-2025.2/Biogenetica/dataset/'
ARQUIVO_ENTRADA = os.path.join(GDRIVE_PATH, 'dataset_FINAL_ACHATADO.csv')

TARGET_FUNCTION = 'protein binding'

# --- Configs da CNN ---
# Vamos "fatiar" todas as sequências nesse comprimento.
# 2000 bases é um bom equilíbrio entre manter o sinal e economizar RAM.
MAX_SEQ_LENGTH = 2000
EMBEDDING_DIM = 100  # Dimensão do vetor para cada base (A,T,C,G)

# --- Configs do Pipeline de Leitura ---
# Vamos ler o arquivo de 14.4M em chunks maiores para acelerar
CHUNK_SIZE = 100000

In [ ]:
print(f"Iniciando Pipeline Final (CNN)...")
print(f"Arquivo de entrada: {ARQUIVO_ENTRADA}") # Este é o dataset_FINAL_ACHATADO.csv
print(f"Tarefa: Classificação Binária (Target = '{TARGET_FUNCTION}')\n")

# --- 3.1 Carregar Dados ---
# (Sem chunks! Vamos carregar o arquivo de ~80k linhas de uma vez)
print("Carregando dataset 'achatado'...")
try:
    df_flat = pd.read_csv(ARQUIVO_ENTRADA)
except FileNotFoundError:
    print(f"ERRO: Arquivo '{ARQUIVO_ENTRADA}' não encontrado.")
    raise
except Exception as e:
    print(f"Erro ao ler CSV: {e}")
    raise

# Verifica se a coluna 'label' (que o script anterior criou) existe
if 'label' not in df_flat.columns or 'Sequencia' not in df_flat.columns:
    print(f"ERRO: O CSV de entrada não contém 'label' ou 'Sequencia'.")
    print("Você rodou o script 'juntar_dados_v2_RAM_Safe.py' primeiro?")
    raise

print(f"Dataset carregado com {len(df_flat)} genes únicos.")

# --- 3.2 Balanceamento (Undersampling) ---
print(f"Distribuição ANTES do balanceamento:\n{df_flat['label'].value_counts()}\n")
print("Iniciando Undersampling para forçar balanço 1:1...")
rus = RandomUnderSampler(random_state=42)

X_para_amostrar = df_flat.index.values.reshape(-1, 1)
y_para_amostrar = df_flat['label']

try:
    X_res, y_res = rus.fit_resample(X_para_amostrar, y_para_amostrar)
except ValueError as e:
    print(f"\nERRO NO BALANCEAMENTO: {e}")
    print("Isso geralmente acontece se uma classe tem 0 amostras.")
    raise

indices_balanceados = X_res.flatten()
df_balanced = df_flat.loc[indices_balanceados].copy()
print(f"Dataset balanceado criado.")
print(f"Distribuição DEPOIS do balanceamento:\n{df_balanced['label'].value_counts()}\n")

# --- 3.3 Preparação para CNN (Tokenizer e Padding) ---
print("Preparando dados para CNN (Tokenizing e Padding)...")

# Garantir que as sequências são strings
df_balanced['Sequencia'] = df_balanced['Sequencia'].astype(str)

tokenizer = Tokenizer(char_level=True, lower=False, oov_token='U') # 'U' para desconhecido
tokenizer.fit_on_texts(df_balanced['Sequencia'])
sequences_tokenized = tokenizer.texts_to_sequences(df_balanced['Sequencia'])

# Atualiza o tamanho do vocabulário (A, T, C, G, N, U, etc.)
VOCAB_SIZE = len(tokenizer.word_index)
print(f"Tamanho do Vocabulário (A,T,C,G,N...): {VOCAB_SIZE}")

X_padded = pad_sequences(sequences_tokenized,
                         maxlen=MAX_SEQ_LENGTH,
                         padding='post',
                         truncating='post')

Y_labels = df_balanced['label'].values

print(f"Matriz de features X criada: {X_padded.shape}")
print(f"Vetor de labels Y criado: {Y_labels.shape}")

# --- 3.4 Divisão de Treino/Teste ---
print("\nDividindo dados (80% treino / 20% teste)...")
X_train, X_test, y_train, y_test = train_test_split(
    X_padded,
    Y_labels,
    test_size=0.2,
    random_state=42,
    stratify=Y_labels
)
print(f"Tamanho do Treino: {X_train.shape[0]}")
print(f"Tamanho do Teste: {X_test.shape[0]}")

Iniciando Pipeline Final (CNN)...
Arquivo de entrada: /content/drive/MyDrive/Faculdade/IC-2025.2/Biogenetica/dataset/dataset_FINAL_ACHATADO.csv
Tarefa: Classificação Binária (Target = 'protein binding')

Carregando dataset 'achatado'...
Dataset carregado com 5014 genes únicos.
Distribuição ANTES do balanceamento:
label
1    3723
0    1291
Name: count, dtype: int64

Iniciando Undersampling para forçar balanço 1:1...
Dataset balanceado criado.
Distribuição DEPOIS do balanceamento:
label
0    1291
1    1291
Name: count, dtype: int64

Preparando dados para CNN (Tokenizing e Padding)...
Tamanho do Vocabulário (A,T,C,G,N...): 16
Matriz de features X criada: (2582, 2000)
Vetor de labels Y criado: (2582,)

Dividindo dados (80% treino / 20% teste)...
Tamanho do Treino: 2065
Tamanho do Teste: 517


In [ ]:
print("Construindo o modelo CNN (Otimizado contra Overfitting)...")

# Usamos +1 no VOCAB_SIZE porque 0 é reservado para o 'padding'
model = Sequential()

# 1. Camada de Embedding
model.add(Embedding(input_dim=VOCAB_SIZE + 1,
                    output_dim=EMBEDDING_DIM,
                    input_length=MAX_SEQ_LENGTH))

# 2. Bloco Convolucional 1
model.add(Conv1D(filters=64, kernel_size=10, activation='relu', padding='same'))
model.add(MaxPooling1D(pool_size=4))

# 3. Bloco Convolucional 2 (Mais profundo)
model.add(Conv1D(filters=128, kernel_size=8, activation='relu', padding='same'))
model.add(MaxPooling1D(pool_size=4))

# 4. Achatamento
# (Usar GlobalMaxPooling é muitas vezes melhor que Flatten para texto)
model.add(GlobalMaxPooling1D())

# 5. Camada Densa (Classificador)
model.add(Dense(256, activation='relu'))
# --- OTIMIZAÇÃO ANTI-OVERFITTING ---
model.add(Dropout(0.5)) # Desliga 50% dos neurônios no treino

# 6. Camada de Saída
model.add(Dense(1, activation='sigmoid'))

# Compila o modelo
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()

Construindo o modelo CNN (Otimizado contra Overfitting)...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
print("\nIniciando treinamento da CNN FINAL...")
print("Usando EarlyStopping: o treino vai parar se a 'val_loss' não melhorar.")

# --- OTIMIZAÇÃO ANTI-OVERFITTING ---
# Se a perda na validação (val_loss) não melhorar por 3 épocas seguidas, pare.
early_stopping = EarlyStopping(monitor='val_loss',
                              patience=3,
                              restore_best_weights=True)

# Vamos treinar (Isso VAI demorar. Vá tomar um café de verdade)
history = model.fit(X_train, y_train,
                    epochs=20, # Máximo de 20, mas o EarlyStopping deve parar antes
                    batch_size=64,
                    validation_data=(X_test, y_test),
                    callbacks=[early_stopping]) # <-- Adiciona o callback

print("Treinamento concluído.")


Iniciando treinamento da CNN FINAL...
Usando EarlyStopping: o treino vai parar se a 'val_loss' não melhorar.
Epoch 1/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 11s 141ms/step - accuracy: 0.5147 - loss: 0.6938 - val_accuracy: 0.5145 - val_loss: 0.6913
Epoch 2/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.5083 - loss: 0.6941 - val_accuracy: 0.5725 - val_loss: 0.6906
Epoch 3/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.5241 - loss: 0.6930 - val_accuracy: 0.4990 - val_loss: 0.6916
Epoch 4/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.5108 - loss: 0.6936 - val_accuracy: 0.5126 - val_loss: 0.6915
Epoch 5/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.5269 - loss: 0.6923 - val_accuracy: 0.5648 - val_loss: 0.6888
Epoch 6/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.5101 - loss: 0.6923 - val_accuracy: 0.5629 - val_loss: 0.6888
Epoch 7/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.5169 - loss: 0.6898 - val_accuracy: 0.5957 - val_loss: 0.6822

In [ ]:
print("\nAvaliando modelo final nos dados de teste...")

y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
print(f"\n--- Relatório de Desempenho (CNN FINAL) ---")
print(f"Acurácia Geral: {acc * 100:.2f}%")

print("\nRelatório de Classificação (Precisão, Recall, F1 por Classe):")
print(classification_report(y_test, y_pred))


Avaliando modelo final nos dados de teste...
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step

--- Relatório de Desempenho (CNN FINAL) ---
Acurácia Geral: 59.19%

Relatório de Classificação (Precisão, Recall, F1 por Classe):
              precision    recall  f1-score   support

           0       0.59      0.58      0.59       259
           1       0.59      0.60      0.60       258

    accuracy                           0.59       517
   macro avg       0.59      0.59      0.59       517
weighted avg       0.59      0.59      0.59       517



In [ ]:
import pandas as pd
import numpy as np
import sys
import os

try:
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, accuracy_score
    from sklearn.preprocessing import StandardScaler # <-- Novo!
    from sklearn.exceptions import UndefinedMetricWarning
    import warnings

    # --- NOVO IMPORT ---
    from imblearn.under_sampling import RandomUnderSampler
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping

except ImportError:
    print("ERRO: Bibliotecas não encontradas.")
    print("Rode: pip install scikit-learn imbalanced-learn pandas numpy tensorflow")
    sys.exit(1)

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

# --- 1. Configurações ---
ARQUIVO_ENTRADA = '/content/drive/MyDrive/Faculdade/IC-2025.2/Biogenetica/dataset/dataset_FINAL_ACHATADO.csv'
KMER_SIZE = 6
TARGET_FUNCTION = 'protein binding'

# --- 2. Função de k-mer ---
def get_kmers_string(sequence, k):
    seq_str = str(sequence)
    kmers = [seq_str[i:i+k] for i in range(len(seq_str) - k + 1)]
    return " ".join(kmers)

# --- 3. Script Principal ---
print(f"Iniciando Protótipo de Classificação (MLP + K-mers)...")
print(f"Arquivo de entrada: {ARQUIVO_ENTRADA}")

# --- 3.1 Carregar Dados ---
print("Carregando dataset 'achatado'...")
try:
    df_flat = pd.read_csv(ARQUIVO_ENTRADA)
except FileNotFoundError:
    print(f"ERRO: Arquivo '{ARQUIVO_ENTRADA}' não encontrado.")
    raise

if 'label' not in df_flat.columns or 'Sequencia' not in df_flat.columns:
    print(f"ERRO: O CSV de entrada não contém 'label' ou 'Sequencia'.")
    raise

print(f"Dataset carregado com {len(df_flat)} genes únicos.")

# --- 3.2 Balanceamento (Undersampling) ---
print(f"Distribuição ANTES do balanceamento:\n{df_flat['label'].value_counts()}\n")
print("Iniciando Undersampling para forçar balanço 1:1...")
rus = RandomUnderSampler(random_state=42)

X_para_amostrar = df_flat.index.values.reshape(-1, 1)
y_para_amostrar = df_flat['label']
X_res, y_res = rus.fit_resample(X_para_amostrar, y_para_amostrar)

indices_balanceados = X_res.flatten()
df_balanced = df_flat.loc[indices_balanceados].copy()
print(f"Dataset balanceado criado.\nDistribuição DEPOIS:\n{df_balanced['label'].value_counts()}\n")

# --- 3.3 Engenharia de Features (K-mers) ---
print(f"Iniciando engenharia de features (k-mers, k={KMER_SIZE})...")
vectorizer = CountVectorizer(analyzer='word')

print("Treinando Vectorizer (fit)...")
vectorizer.fit(df_flat['Sequencia'].astype(str).apply(lambda x: get_kmers_string(x, KMER_SIZE)))
print("Transformando dados balanceados (transform)...")
X_kmers = vectorizer.transform(df_balanced['Sequencia'].astype(str).apply(lambda x: get_kmers_string(x, KMER_SIZE)))
Y_labels = df_balanced['label']

print(f"Matriz de features X criada: {X_kmers.shape}")

# --- 3.4 Normalização (Novo!) ---
# Redes neurais ODEIAM contagens (0, 1, 5, 50). Elas preferem números pequenos (ex: -1 a 1).
# O StandardScaler faz isso.
print("Normalizando dados (StandardScaler)...")
scaler = StandardScaler(with_mean=False) # with_mean=False é melhor para dados esparsos (sparse)
X_scaled = scaler.fit_transform(X_kmers)

# --- 3.5 Treinamento (MLP) ---
print("\nDividindo dados (80% treino / 20% teste)...")
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, # Usamos os dados normalizados
    Y_labels,
    test_size=0.2,
    random_state=42,
    stratify=Y_labels
)

print(f"Iniciando treinamento da Rede Neural Densa (MLP) em {X_train.shape[0]} amostras...")

# --- Definição do Modelo MLP ---
model = Sequential()
model.add(Dense(128, input_dim=X_train.shape[1], activation='relu')) # 4110 inputs
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid')) # Saída binária

model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()

# Callback de parada
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Treinamento (convertendo X para denso, pois o Keras prefere)
history = model.fit(X_train.toarray(), y_train,
                    epochs=20,
                    batch_size=64,
                    validation_data=(X_test.toarray(), y_test),
                    callbacks=[early_stopping])

print("Treinamento concluído.")

# --- 3.6 Avaliação Final ---
print("\nAvaliando modelo nos dados de teste...")
y_pred_proba = model.predict(X_test.toarray())
y_pred = (y_pred_proba > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
print(f"\n--- Relatório de Desempenho (MLP + K-mers) ---")
print(f"Acurácia Geral: {acc * 100:.2f}%")

print("\nRelatório de Classificação (Precisão, Recall, F1 por Classe):")
print(classification_report(y_test, y_pred))

print("\n\nTreinamento e Avaliação do Modelo Final CONCLUÍDOS.")